## Data setup 

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset.kitti_dataset import KittiDataset
from src.preprocessing.camera_bev import aggregate_rgb_points_to_bev
from src.geometry.camera_projection import project_lidar_to_image
from src.preprocessing.bev import bev_projection

In [2]:
dataset = KittiDataset(
    root=PROJECT_ROOT / "data" / "KITTI",
    split="training",
    load_images=True,
)

sample = dataset[0]

In [3]:
# fusion bev

def fused_bev(points, image, calib):
    pixels, valid_indices, depths = project_lidar_to_image(points, calib, image.shape)

    visible_lidar_points = points[valid_indices]

    # Nearest-pixel RGB sampling for the fusion prototype.
    pixel_u = np.rint(
        pixels[:, 0]
    ).astype(np.int64)

    pixel_v = np.rint(
        pixels[:, 1]
    ).astype(np.int64)

    # Rounding can move an edge coordinate by one pixel.
    pixel_u = np.clip(
        pixel_u,
        0,
        image.shape[1] - 1,
    )
    pixel_v = np.clip(
        pixel_v,
        0,
        image.shape[0] - 1,
    )

    rgb = (
        image[pixel_v, pixel_u]
        .astype(np.float32)
        / 255.0
    )

    fused_points = np.concatenate(
        [
            visible_lidar_points,
            rgb,
        ],
        axis=1,
    )

    return fused_points

In [4]:
fused_points = fused_bev(sample["points"], sample["image"], sample["calib"])

In [5]:
fused_points.shape

(20285, 7)